# Danenberg 2022: cell tables, masks, clinical data and a frozen patient split

This notebook builds the cell-level data that the Kaggle dataset `yuvalsamoilov/danenberg2022` lacks; that dataset holds only images and the panel.
The source is the public release on Zenodo, record 5850952, `METABRICBreastCancerIMCTME.zip`.
Members are read with HTTP range requests, so the 6.2 GB zip is never downloaded in full.

Outputs, written to `/kaggle/working`. Later kernels attach them via `kernel_sources`.
- `cells.parquet`: one row per cell.
  - Channel columns use the `panel.parquet` target names and hold mean isotopic counts, spillover-corrected as published.
  - Other columns: x/y (µm = px), area, phenotype and the flags.
- `cell_neighbours.parquet`: the published contact graph (`ImageNumber`, `from`, `to`).
- `masks/<image_name>.tiff`: the published CellMask, **relabelled to `ObjectNumber`**.
  - The published masks store rescaled labels (0–65535), so labels are matched to table cells by nearest centroid.
- `clinical.parquet`: METABRIC clinical data from the cBioPortal API (`brca_metabric`).
- `danenberg_patient_split.csv`: the **frozen** patient-level train/val/test split (60/20/20), stratified by ER, HER2 and PAM50/claudin subtype.
- `prep_report.json`: every check listed in the last cell.

In [ ]:
# The Kaggle images are ZSTD-compressed TIFFs. This imagecodecs build works with the image's numpy 2.0.
!pip install -q "imagecodecs==2024.9.22"

In [ ]:
import os, io, json, glob, time, zipfile
import numpy as np, pandas as pd, requests, tifffile
from scipy.spatial import cKDTree

ZENODO = "https://zenodo.org/records/5850952/files/METABRICBreastCancerIMCTME.zip?download=1"
ROOT = "toPublicRepository"
OUT = "/kaggle/working"
os.makedirs(f"{OUT}/masks", exist_ok=True)
SEED = 20260927
REPORT = {}

def find_input(pattern):
    """Kaggle has used several mount layouts; search all of /kaggle/input."""
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

img_files = find_input("MB_*.tiff")
panel = pd.read_parquet(find_input("panel.parquet")[0])
img_by_name = {os.path.splitext(os.path.basename(p))[0]: p for p in img_files}
print(f"{len(img_files)} Kaggle images, panel {panel.shape}")

In [ ]:
class HTTPRangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP range requests (with retries)."""
    def __init__(self, url):
        self.s = requests.Session()
        r = self.s.head(url, allow_redirects=True, timeout=60)
        self.url, self.size, self.pos = r.url, int(r.headers["Content-Length"]), 0
    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self.pos
    def seek(self, off, whence=0):
        self.pos = {0: off, 1: self.pos + off, 2: self.size + off}[whence]
        return self.pos
    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0:
            return 0
        for attempt in range(8):
            try:
                r = self.s.get(self.url, timeout=120,
                               headers={"Range": f"bytes={self.pos}-{self.pos + n - 1}"})
                r.raise_for_status()
                data = r.content
                break
            except Exception as e:
                print(f"  range retry {attempt}: {e}")
                time.sleep(2 ** attempt)
        else:
            raise IOError("range request failed")
        b[:len(data)] = data
        self.pos += len(data)
        return len(data)

def download(url, path, chunk=16 << 20):
    """One streaming GET (resumed with Range if the connection drops). Many small range
    requests were throttled by Zenodo (version 3 of this kernel ran for hours)."""
    size = int(requests.head(url, allow_redirects=True, timeout=60).headers["Content-Length"])
    t0 = time.time()
    for attempt in range(10):
        have = os.path.getsize(path) if os.path.exists(path) else 0
        if have >= size:
            break
        try:
            with requests.get(url, stream=True, timeout=300,
                              headers={"Range": f"bytes={have}-"} if have else {}) as r:
                r.raise_for_status()
                with open(path, "ab") as f:
                    for i, b in enumerate(r.iter_content(chunk)):
                        f.write(b)
                        if i % 40 == 0:
                            got = os.path.getsize(path)
                            print(f"  {got / 1e9:.2f}/{size / 1e9:.2f} GB  {got / 1e6 / (time.time() - t0 + 1e-9):.0f} MB/s")
        except Exception as e:
            print(f"  download retry {attempt}: {e}")
            time.sleep(min(60, 2 ** attempt))
    assert os.path.getsize(path) == size, "incomplete download"
    return path

TMP = next(d for d in ["/kaggle/tmp", "/tmp"] if os.path.isdir(d))
import shutil
free = shutil.disk_usage(TMP).free
print(f"free space in {TMP}: {free / 1e9:.1f} GB")
if free > 8e9:
    zf = zipfile.ZipFile(download(ZENODO, f"{TMP}/mbtme.zip"))
    zfm = zf
else:  # fallback: range reads; a 1 MB buffer for the many small mask members
    zf = zipfile.ZipFile(io.BufferedReader(HTTPRangeFile(ZENODO), buffer_size=16 << 20))
    zfm = zipfile.ZipFile(io.BufferedReader(HTTPRangeFile(ZENODO), buffer_size=1 << 20))
names = set(zf.namelist())
print(len(names), "zip members")

## Single-cell table
Marker columns follow the stack order, which matches `panel.parquet` (`channel_index`). They are renamed by metal, via `markerStackOrder.csv`.

In [ ]:
t0 = time.time()
sc = pd.read_csv(zf.open(f"{ROOT}/SingleCells.csv"))
print(sc.shape, f"{time.time() - t0:.0f}s")

mso = pd.read_csv(zf.open(f"{ROOT}/markerStackOrder.csv"))
first, last = sc.columns.get_loc("is_hotAggregate") + 1, sc.columns.get_loc("Location_Center_X")
marker_cols = list(sc.columns[first:last])
# Map by position: the stack order (metals) must equal the panel order. Names can differ slightly,
# e.g. markerStackOrder says 'c-Caspase3c-PARP' where SingleCells says 'c-Caspase3'.
panel_sorted = panel.sort_index()
assert len(marker_cols) == len(mso) == len(panel_sorted) == 39
assert list(mso.Isotope) == list(panel_sorted.metal_mass), "stack metals != panel metals"
REPORT["name_mismatches_table_vs_stackorder"] = [(a, b) for a, b in zip(marker_cols, mso.Epitope) if a != b]
epi2target = dict(zip(marker_cols, panel_sorted.target))

cells = pd.DataFrame({
    "image_number": sc.ImageNumber.astype(np.int32),
    "object_number": sc.ObjectNumber.astype(np.int32),
    "metabric_id": sc.metabric_id.astype(str),
    "image_name": sc.metabric_id.str.replace("-", "_") + "_" + sc.ImageNumber.astype(str),
    "cell_phenotype": sc.cellPhenotype.astype(str),
    "x": sc.Location_Center_X.astype(np.float32),
    "y": sc.Location_Center_Y.astype(np.float32),
    "area": sc.AreaShape_Area.astype(np.float32),
})
for c in ["is_epithelial", "is_tumour", "is_normal", "is_dcis", "is_interface",
          "is_perivascular", "is_hotAggregate"]:
    cells[c.replace("hotAggregate", "hot_aggregate")] = sc[c].astype(np.int8)
for e in marker_cols:
    cells[epi2target[e]] = sc[e].astype(np.float32)
del sc

REPORT["n_cells"] = int(len(cells))
REPORT["n_images_table"] = int(cells.image_name.nunique())
REPORT["n_patients_table"] = int(cells.metabric_id.nunique())
REPORT["n_tumour_images"] = int(cells.loc[cells.is_tumour == 1, "image_name"].nunique())
REPORT["images_missing_on_kaggle"] = sorted(set(cells.image_name) - set(img_by_name))
REPORT["kaggle_images_not_in_table"] = sorted(set(img_by_name) - set(cells.image_name))
REPORT["channel_map"] = epi2target
REPORT["phenotypes"] = cells.cell_phenotype.value_counts().to_dict()
chan = list(epi2target.values())
REPORT["channel_quantiles"] = {c: np.round(np.quantile(cells[c], [0.01, 0.5, 0.99]), 3).tolist() for c in chan}
cells.to_parquet(f"{OUT}/cells.parquet", index=False)
print({k: REPORT[k] for k in ["n_cells", "n_images_table", "n_patients_table", "n_tumour_images"]})
print("missing on kaggle:", len(REPORT["images_missing_on_kaggle"]),
      "| kaggle not in table:", len(REPORT["kaggle_images_not_in_table"]))

## Contact graph

In [ ]:
t0 = time.time()
nb = pd.read_csv(zf.open(f"{ROOT}/CellNeighbours.csv"), usecols=["ImageNumber", "from", "to"],
                 dtype={"ImageNumber": np.int32, "from": np.int32, "to": np.int32})
nb.to_parquet(f"{OUT}/cell_neighbours.parquet", index=False)
REPORT["n_contacts"] = int(len(nb))
print(nb.shape, f"{time.time() - t0:.0f}s")
del nb

## Clinical data (cBioPortal `brca_metabric`)

In [ ]:
base = "https://www.cbioportal.org/api/studies/brca_metabric/clinical-data"
parts = []
for t in ["PATIENT", "SAMPLE"]:
    r = requests.get(base, params={"clinicalDataType": t, "projection": "SUMMARY",
                                   "pageSize": 10_000_000}, timeout=300)
    r.raise_for_status()
    d = pd.DataFrame(r.json())
    parts.append(d.pivot_table(index="patientId", columns="clinicalAttributeId",
                               values="value", aggfunc="first"))
clin = parts[0].join(parts[1], rsuffix="_sample")
clin.index.name = "metabric_id"
clin = clin.reset_index()
clin.to_parquet(f"{OUT}/clinical.parquet", index=False)
pats = sorted(cells.metabric_id.unique())
cov = clin.set_index("metabric_id").reindex(pats)
REPORT["clinical_coverage"] = {c: float(cov[c].notna().mean()) for c in
                               ["ER_STATUS", "HER2_STATUS", "CLAUDIN_SUBTYPE", "OS_MONTHS"]}
REPORT["patients_without_clinical"] = sorted(cov.index[cov.ER_STATUS.isna()])
print(clin.shape, REPORT["clinical_coverage"], len(REPORT["patients_without_clinical"]), "patients without clinical data")

## Frozen patient split (60/20/20)
Strata are ER × HER2 × PAM50/claudin subtype, with missing values as their own level. Strata with fewer than 5 patients are pooled.
Within each stratum, patients are shuffled with a fixed seed.

In [ ]:
s = cov[["ER_STATUS", "HER2_STATUS", "CLAUDIN_SUBTYPE"]].fillna("NA")
stratum = s.ER_STATUS + "|" + s.HER2_STATUS + "|" + s.CLAUDIN_SUBTYPE
small = stratum.map(stratum.value_counts()) < 5
stratum.loc[small] = "pooled"
rng = np.random.default_rng(SEED)
split = pd.Series(index=stratum.index, dtype=object)
for g, idx in stratum.groupby(stratum).groups.items():
    idx = np.array(sorted(idx))
    rng.shuffle(idx)
    n = len(idx)
    n_tr, n_va = int(round(0.6 * n)), int(round(0.2 * n))
    split.loc[idx[:n_tr]] = "train"
    split.loc[idx[n_tr:n_tr + n_va]] = "val"
    split.loc[idx[n_tr + n_va:]] = "test"
split_df = pd.DataFrame({"metabric_id": stratum.index, "split": split.values,
                         "stratum": stratum.values, "er": s.ER_STATUS.values,
                         "her2": s.HER2_STATUS.values, "pam50": s.CLAUDIN_SUBTYPE.values})
split_df.to_csv(f"{OUT}/danenberg_patient_split.csv", index=False)
REPORT["split_sizes"] = split_df.split.value_counts().to_dict()
REPORT["split_by_er"] = pd.crosstab(split_df.split, split_df.er).to_dict()
print(REPORT["split_sizes"])

## Masks: relabel to `ObjectNumber`, check against the Kaggle images
- Each published label is matched to its table cell by nearest centroid. The match must be under 2 px and one-to-one.
- Mask labels with no table cell (e.g. cells removed as hot aggregates) become 0.
- Alignment check on 25 random images: per-cell means computed from the Kaggle image with the relabelled mask must correlate with the published table values for H3, DNA1, DNA2, panCK and ER.

In [ ]:
def read_img(path):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == len(panel) and im.shape[-1] != len(panel):
        im = np.moveaxis(im, 0, -1)
    return im  # H, W, C

cells_by_img = {k: g for k, g in cells.groupby("image_name")}
rng = np.random.default_rng(SEED)
check_imgs = set(rng.choice(sorted(cells_by_img), min(25, len(cells_by_img)), replace=False))
chk_chan = ["histone_h3", "dna1", "dna2", "pan_cytokeratin", "estrogen_receptor_alpha"]
stats, align = [], []
t0 = time.time()
for i, (name, g) in enumerate(sorted(cells_by_img.items())):
    mid, num = name.rsplit("_", 1)
    member = f"{ROOT}/Images/{mid.replace('_', '')}_{num}_CellMask.tiff"
    m = tifffile.imread(io.BytesIO(zfm.read(member))).astype(np.int64)
    u, inv = np.unique(m, return_inverse=True)
    inv = inv.reshape(m.shape)
    ys, xs = np.indices(m.shape)
    cnt = np.bincount(inv.ravel())
    cx = np.bincount(inv.ravel(), xs.ravel()) / cnt
    cy = np.bincount(inv.ravel(), ys.ravel()) / cnt
    fg = np.where(u > 0)[0]
    dist, j = cKDTree(np.c_[cx[fg], cy[fg]]).query(g[["x", "y"]].values)
    lut = np.zeros(len(u), np.int64)
    ok = dist < 2.0
    lut[fg[j[ok]]] = g.object_number.values[ok]
    relab = lut[inv].astype(np.uint16)
    tifffile.imwrite(f"{OUT}/masks/{name}.tiff", relab, compression="zlib")
    st = dict(image_name=name, h=m.shape[0], w=m.shape[1], n_table=len(g), n_mask=len(fg),
              n_matched=int(ok.sum()), n_unique=int(len(np.unique(j[ok]))),
              dist_max=float(dist.max()))
    if name in img_by_name:
        im_shape = tifffile.TiffFile(img_by_name[name]).series[0].shape
        hw = im_shape[1:] if im_shape[0] == len(panel) else im_shape[:2]
        st["shape_match"] = tuple(hw) == m.shape
        if name in check_imgs and st["shape_match"]:
            im = read_img(img_by_name[name])
            lab = relab.astype(np.int64).ravel()
            n_obj = int(g.object_number.max()) + 1
            area = np.bincount(lab, minlength=n_obj)
            gi = g.set_index("object_number")
            for c in chk_chan:
                ci = int(panel.index[panel.target == c][0])
                mean = np.bincount(lab, im[..., ci].ravel().astype(np.float64), n_obj) / np.maximum(area, 1)
                o = gi.index.values
                align.append(dict(image_name=name, channel=c,
                                  r=float(np.corrcoef(mean[o], gi[c].values)[0, 1])))
    else:
        st["shape_match"] = None
    stats.append(st)
    if i % 100 == 0:
        print(f"  {i}/{len(cells_by_img)}  {time.time() - t0:.0f}s")

ms = pd.DataFrame(stats)
ms.to_csv(f"{OUT}/mask_stats.csv", index=False)
al = pd.DataFrame(align, columns=["image_name", "channel", "r"])
REPORT["masks_written"] = int(len(ms))
REPORT["mask_shape_match_frac"] = float(ms.shape_match.fillna(False).astype(bool).mean())
REPORT["mask_cells_matched_frac"] = float(ms.n_matched.sum() / ms.n_table.sum())
REPORT["mask_one_to_one"] = bool((ms.n_unique == ms.n_matched).all())
REPORT["mask_dist_max"] = float(ms.dist_max.max())
REPORT["alignment_r_median_by_channel"] = al.groupby("channel").r.median().round(4).to_dict()
REPORT["alignment_r_min"] = float(al.r.min()) if len(al) else None
print(json.dumps({k: REPORT[k] for k in REPORT if k.startswith(("mask", "alignment"))}, indent=1))

## Report and hard checks

In [ ]:
checks = {
    "all_table_images_on_kaggle": len(REPORT["images_missing_on_kaggle"]) == 0,
    "mask_shapes_match": REPORT["mask_shape_match_frac"] == 1.0,
    "mask_match_>=99%": REPORT["mask_cells_matched_frac"] >= 0.99,
    "mask_one_to_one": REPORT["mask_one_to_one"],
    "alignment_dna1_r>0.9": REPORT["alignment_r_median_by_channel"].get("dna1", 0) > 0.9,
    # 107/718 Danenberg patients are not in cBioPortal brca_metabric (coverage 0.85); they keep an "NA" stratum.
    "clinical_er_coverage>=0.8": REPORT["clinical_coverage"]["ER_STATUS"] >= 0.8,
}
REPORT["checks"] = checks
with open(f"{OUT}/prep_report.json", "w") as f:
    json.dump(REPORT, f, indent=1, default=str)
print(json.dumps(checks, indent=1))
assert all(checks.values()), "prep checks failed - see prep_report.json"